In [5]:
import pandas as pd
import networkx as nx
import os
from dotenv import load_dotenv, find_dotenv
import igraph as ig

In [6]:
load_dotenv(find_dotenv())
root = os.getenv("ROOT_DIR")

graph_df = pd.read_csv(os.path.join(root, 'data', 'cleaned_collaboration.csv'))

In [7]:
print(graph_df.shape)
print(graph_df.columns)

(188305, 14)
Index(['id_1', 'id_2', 'name_1', 'followers_1', 'popularity_1', 'genres_1',
       'chart_hits_1', 'top_chart_1', 'name_2', 'followers_2', 'popularity_2',
       'genres_2', 'chart_hits_2', 'top_chart_2'],
      dtype='object')


In [12]:
import os, math
import pandas as pd
import igraph as ig
from tqdm import tqdm

def build_openord_graph(df: pd.DataFrame, root: str = "."):
    # ---------- 1) collect unique node attributes ----------
    node_attrs, dropped_idx = {}, set()
    for side in (1, 2):
        id_col = f"id_{side}"
        cols   = {k: f"{k}_{side}" for k in
                  ("name", "followers", "popularity",
                   "genres", "chart_hits", "top_chart")}

        # wrap the row iteration in tqdm
        for idx, row in tqdm(df.iterrows(),
                             total=len(df),
                             desc=f"Collect attrs side {side}",
                             unit="row"):
            if idx in dropped_idx:
                continue
            nid   = row[id_col]
            attrs = {k: row[v] for k, v in cols.items()}
            if nid in node_attrs:
                for k, v in attrs.items():
                    if k in ("chart_hits", "top_chart"):
                        continue
                    if node_attrs[nid][k] != v:
                        dropped_idx.add(idx)
                        break
            else:
                node_attrs[nid] = attrs

    # ---------- 2) build edge list (skip conflicting rows) ----------
    ids    = list(node_attrs)
    id2idx = {nid: i for i, nid in enumerate(ids)}
    edges  = []
    for i, row in tqdm(df.iterrows(),
                       total=len(df),
                       desc="Building edges",
                       unit="row"):
        if i in dropped_idx:
            continue
        edges.append((id2idx[row.id_1], id2idx[row.id_2]))

    # ---------- 3) create igraph ----------
    g = ig.Graph(n=len(ids), edges=edges, directed=False)
    for attr in ("name", "followers", "popularity",
                 "genres", "chart_hits", "top_chart"):
        g.vs[attr] = [node_attrs[n][attr] for n in ids]
    g.vs["degree"] = g.degree()

    # ---------- 4) compute layout ----------
    main_ids = [v.index for v in g.vs if v["degree"] > 0]
    iso_ids  = [v.index for v in g.vs if v["degree"] == 0]
    coords   = [[math.nan, math.nan] for _ in range(g.vcount())]

    if main_ids:
        sub = g.induced_subgraph(main_ids)
        try:
            lay = sub.layout("openord", dim=2)
        except (KeyError, ig.InternalError):
            lay = sub.layout("lgl")
        for loc, glob in enumerate(main_ids):
            coords[glob] = lay[loc]

    # place isolates on a circle — wrap in tqdm for feedback
    if iso_ids:
        r = max(max(abs(x), abs(y)) for x, y in coords
                if not math.isnan(x)) or 1.0
        for k, vid in tqdm(enumerate(iso_ids),
                           total=len(iso_ids),
                           desc="Positioning isolates",
                           unit="iso"):
            phi        = 2 * math.pi * k / len(iso_ids)
            coords[vid] = (r*1.05*math.cos(phi),
                           r*1.05*math.sin(phi))

    # ---------- 5) export ----------
    nodes_df = (
        pd.DataFrame.from_dict(node_attrs, orient="index")
          .reset_index().rename(columns={"index": "id"})
          .assign(x=[c[0] for c in coords],
                  y=[c[1] for c in coords],
                  degree=g.vs["degree"])
    )
    # nodes_df.to_csv(os.path.join(root, "data", "network_nodes.csv"),
    #                 index=False)

    edges_df = (pd.DataFrame(edges, columns=["src_idx", "tgt_idx"])
                  .assign(source=lambda d: d.src_idx.map(lambda i: ids[i]),
                          target=lambda d: d.tgt_idx.map(lambda i: ids[i]))
                  .loc[:, ["source", "target"]])
    # edges_df.to_csv(os.path.join(root, "data", "network_edges.csv"),
    #                 index=False)

    return g, nodes_df, edges_df


In [13]:
g, nodes, edges = build_openord_graph(graph_df)



Building edges: 100%|██████████| 188305/188305 [00:08<00:00, 21564.34row/s]
c:\Python312\Lib\site-packages\igraph\layout.py:691: RuntimeWarning: LGL layout does not support disconnected graphs yet. at src/layout/large_graph.c:179
  layout = func(*args, **kwds)


In [14]:
g.to_csv(os.path.join(root, 'data', 'constructed_network.csv'))

AttributeError: 'Graph' object has no attribute 'to_csv'

In [15]:
nodes.to_csv(os.path.join(root, 'data', 'network_nodes.csv'))
edges.to_csv(os.path.join(root, 'data', 'network_edges.csv'))